In [6]:
# LOCAL NOTEBOOK CELL
# Queries local MusicBrainz PostgreSQL DB for release year, release type,
# label, catalog number, artist type, artist country, and genre tags.
# Applies umbrella genre mapping and joins to album_catalog.parquet.
# Output: album_catalog_enriched.parquet — upload this to Google Drive.

import re
from pathlib import Path

import psycopg2
import psycopg2.extras
import pandas as pd

DATA_DIR = Path("../../datasets")
CATALOG_PATH = DATA_DIR / "album_catalog.parquet"
ENRICHED_PATH = DATA_DIR / "album_catalog_enriched.parquet"

# ── Normalization ────────────────────────────────────────────────────────────
def normalize(s):
    if not isinstance(s, str):
        return ""
    s = s.lower().strip()
    s = re.sub(r"[^a-z0-9 ]", "", s)
    return re.sub(r"\s+", " ", s).strip()

# ── Load album catalog ───────────────────────────────────────────────────────
album_catalog = pd.read_parquet(CATALOG_PATH)
print(f"Albums to enrich: {len(album_catalog):,}")

artist_norms = album_catalog["artist_norm"].dropna().unique().tolist()

# ── MusicBrainz queries ──────────────────────────────────────────────────────
conn = None
try:
    conn = psycopg2.connect(
        dbname="musicbrainz_db",
        user="musicbrainz",
        password="musicbrainz",
        host="localhost",
        port=5432
    )
    cur = conn.cursor()
    cur.execute("SET search_path TO musicbrainz")
    cur.execute("SET statement_timeout = '600s'")

    cur.execute("CREATE TEMP TABLE target_artists (name_norm TEXT PRIMARY KEY)")
    psycopg2.extras.execute_values(
        cur,
        "INSERT INTO target_artists VALUES %s ON CONFLICT DO NOTHING",
        [(n,) for n in artist_norms]
    )
    conn.commit()

    # ── Query 1: Core metadata ───────────────────────────────────────────────
    cur.execute("SET statement_timeout = '600s'")
    cur.execute("""
        SELECT
            rg.gid::text                AS mb_release_group_mbid,
            rg.name                     AS mb_album_name,
            rgm.first_release_date_year AS release_year,
            rgpt.name                   AS release_type,
            ac.name                     AS mb_artist_name,
            a.gid::text                 AS mb_artist_mbid,
            atype.name                  AS artist_type,
            ar.name                     AS artist_area
        FROM release_group rg
        LEFT JOIN release_group_meta rgm
             ON rg.id = rgm.id
        JOIN artist_credit ac
             ON rg.artist_credit = ac.id
        JOIN artist_credit_name acn
             ON ac.id = acn.artist_credit AND acn.position = 0
        JOIN artist a
             ON acn.artist = a.id
        LEFT JOIN artist_type atype
             ON a.type = atype.id
        LEFT JOIN area ar
             ON a.area = ar.id
        LEFT JOIN release_group_primary_type rgpt
             ON rg.type = rgpt.id
        WHERE REGEXP_REPLACE(LOWER(ac.name), '[^a-z0-9 ]', '', 'g')
              IN (SELECT name_norm FROM target_artists)
    """)
    cols = [d[0] for d in cur.description]
    df_mb = pd.DataFrame(cur.fetchall(), columns=cols)
    print(f"Release groups found: {len(df_mb):,}")

    # ── Query 2: Most common label + catalog number per release group ────────
    cur.execute("SET statement_timeout = '600s'")
    cur.execute("""
        SELECT
            rg.gid::text   AS mb_release_group_mbid,
            l.name         AS label,
            rl.catalog_number
        FROM release_group rg
        JOIN artist_credit ac
             ON rg.artist_credit = ac.id
        JOIN release r
             ON r.release_group = rg.id
        JOIN release_label rl
             ON rl.release = r.id
        JOIN label l
             ON rl.label = l.id
        WHERE REGEXP_REPLACE(LOWER(ac.name), '[^a-z0-9 ]', '', 'g')
              IN (SELECT name_norm FROM target_artists)
    """)
    df_labels_raw = pd.DataFrame(
        cur.fetchall(),
        columns=["mb_release_group_mbid", "label", "catalog_number"]
    )
    df_label_agg = (
        df_labels_raw
        .groupby("mb_release_group_mbid")
        .agg(
            label          = ("label",          lambda x: x.value_counts().index[0]),
            catalog_number = ("catalog_number", "first")
        )
        .reset_index()
    )

    # ── Query 3: Genre tags ordered by community vote count ──────────────────
    cur.execute("SET statement_timeout = '600s'")
    cur.execute("""
        SELECT
            rg.gid::text                              AS mb_release_group_mbid,
            array_agg(t.name ORDER BY rgt.count DESC) AS mb_tags
        FROM release_group rg
        JOIN artist_credit ac
             ON rg.artist_credit = ac.id
        JOIN release_group_tag rgt
             ON rg.id = rgt.release_group
        JOIN tag t
             ON rgt.tag = t.id
        WHERE rgt.count > 0
          AND REGEXP_REPLACE(LOWER(ac.name), '[^a-z0-9 ]', '', 'g')
              IN (SELECT name_norm FROM target_artists)
        GROUP BY rg.gid
    """)
    df_tags = pd.DataFrame(
        cur.fetchall(),
        columns=["mb_release_group_mbid", "mb_tags"]
    )
    print(f"Release groups with tags: {len(df_tags):,}")

finally:
    if conn:
        conn.close()

# ── Assemble MB lookup ───────────────────────────────────────────────────────
df_mb = df_mb.merge(df_label_agg, on="mb_release_group_mbid", how="left")
df_mb = df_mb.merge(df_tags,      on="mb_release_group_mbid", how="left")

df_mb["mb_artist_norm"] = df_mb["mb_artist_name"].apply(normalize)
df_mb["mb_album_norm"]  = df_mb["mb_album_name"].apply(normalize)

df_mb = (
    df_mb.sort_values("release_year", na_position="last")
         .drop_duplicates(subset=["mb_artist_norm", "mb_album_norm"])
)

# ── Umbrella genre mapping ───────────────────────────────────────────────────
musicbrainz_to_major_genre = {
    # ROCK
    'rock': 'Rock', 'rock and roll': 'Rock', 'classic rock': 'Rock',
    'hard rock': 'Rock', 'soft rock': 'Rock', 'arena rock': 'Rock',
    'heartland rock': 'Rock', 'mainstream rock': 'Rock', 'southern rock': 'Rock',
    'roots rock': 'Rock', 'swamp rock': 'Rock', 'pub rock': 'Rock',
    'garage rock': 'Rock', 'garage rock revival': 'Rock', 'garage psych': 'Rock',
    'psychedelic rock': 'Rock', 'psychedelic': 'Rock', 'acid rock': 'Rock',
    'raga rock': 'Rock', 'heavy psych': 'Rock', 'progressive rock': 'Rock',
    'progressive': 'Rock', 'prog rock': 'Rock', 'symphonic rock': 'Rock',
    'symphonic prog': 'Rock', 'crossover prog': 'Rock', 'krautrock': 'Rock',
    'space rock': 'Rock', 'glam rock': 'Rock', 'glam': 'Rock',
    'blues rock': 'Rock', 'boogie rock': 'Rock', 'surf rock': 'Rock',
    'surf': 'Rock', 'instrumental rock': 'Rock', 'post-rock': 'Rock',
    'noise rock': 'Rock', 'alternative rock': 'Rock', 'indie rock': 'Rock',
    'grunge': 'Rock', 'post-grunge': 'Rock', 'britpop': 'Rock',
    'post-britpop': 'Rock', 'shoegaze': 'Rock', 'dream pop': 'Rock',
    'jangle pop': 'Rock', 'paisley underground': 'Rock', 'neo-psychedelia': 'Rock',
    'lo-fi': 'Rock', 'slacker rock': 'Rock', 'geek rock': 'Rock',
    'merseybeat': 'Rock', 'beat music': 'Rock', 'freakbeat': 'Rock',
    'mod': 'Rock', 'nederbeat': 'Rock', 'nederpop': 'Rock',
    'frat rock': 'Rock', 'yacht rock': 'Rock', 'aor': 'Rock',
    'industrial rock': 'Rock', 'electronic rock': 'Rock', 'dance-rock': 'Rock',
    'folk rock': 'Rock', 'celtic rock': 'Rock', 'medieval rock': 'Rock',
    'country rock': 'Rock', 'reggae rock': 'Rock', 'tropical rock': 'Rock',
    'stoner rock': 'Rock', 'desert rock': 'Rock', 'occult rock': 'Rock',
    'rockabilly': 'Rock', 'psychobilly': 'Rock', 'acoustic rock': 'Rock',
    'piano rock': 'Rock', 'vocal surf': 'Rock', 'jam band': 'Rock',
    'steampunk': 'Rock', 'rock opera': 'Rock', 'comedy rock': 'Rock',
    'j-rock': 'Rock', 'visual kei': 'Rock', 'gothic': 'Rock',
    'gothic rock': 'Rock', 'funk rock': 'Rock', 'rap rock': 'Rock',
    'slowcore': 'Rock', 'british folk rock': 'Rock', 'canterbury scene': 'Rock',
    # ELECTRONIC/DANCE
    'electronic': 'Electronic/Dance', 'electronica': 'Electronic/Dance',
    'dance': 'Electronic/Dance', 'club': 'Electronic/Dance', 'edm': 'Electronic/Dance',
    'rave': 'Electronic/Dance', 'house': 'Electronic/Dance', 'deep house': 'Electronic/Dance',
    'tech house': 'Electronic/Dance', 'progressive house': 'Electronic/Dance',
    'electro house': 'Electronic/Dance', 'acid house': 'Electronic/Dance',
    'afro house': 'Electronic/Dance', 'funky house': 'Electronic/Dance',
    'vocal house': 'Electronic/Dance', 'french house': 'Electronic/Dance',
    'italo house': 'Electronic/Dance', 'dutch house': 'Electronic/Dance',
    'euro house': 'Electronic/Dance', 'tropical house': 'Electronic/Dance',
    'big room house': 'Electronic/Dance', 'festival progressive house': 'Electronic/Dance',
    'ambient house': 'Electronic/Dance', 'techno': 'Electronic/Dance',
    'ambient techno': 'Electronic/Dance', 'trance': 'Electronic/Dance',
    'progressive trance': 'Electronic/Dance', 'vocal trance': 'Electronic/Dance',
    'hard trance': 'Electronic/Dance', 'big room trance': 'Electronic/Dance',
    'euro-trance': 'Electronic/Dance', 'goa trance': 'Electronic/Dance',
    'psytrance': 'Electronic/Dance', 'psybient': 'Electronic/Dance',
    'dubstep': 'Electronic/Dance', 'brostep': 'Electronic/Dance',
    'future garage': 'Electronic/Dance', 'uk garage': 'Electronic/Dance',
    'uk funky': 'Electronic/Dance', 'drum and bass': 'Electronic/Dance',
    'jungle': 'Electronic/Dance', 'liquid funk': 'Electronic/Dance',
    'neurofunk': 'Electronic/Dance', 'drill and bass': 'Electronic/Dance',
    'jungle terror': 'Electronic/Dance', 'breakbeat': 'Electronic/Dance',
    'breakbeat hardcore': 'Electronic/Dance', 'breakcore': 'Electronic/Dance',
    'big beat': 'Electronic/Dance', 'happy hardcore': 'Electronic/Dance',
    'ambient': 'Electronic/Dance', 'dark ambient': 'Electronic/Dance',
    'drone': 'Electronic/Dance', 'downtempo': 'Electronic/Dance',
    'trip hop': 'Electronic/Dance', 'chillout': 'Electronic/Dance',
    'chillwave': 'Electronic/Dance', 'idm': 'Electronic/Dance',
    'glitch': 'Electronic/Dance', 'glitch hop': 'Electronic/Dance',
    'wonky': 'Electronic/Dance', 'electro': 'Electronic/Dance',
    'disco': 'Electronic/Dance', 'nu disco': 'Electronic/Dance',
    'italo disco': 'Electronic/Dance', 'euro-disco': 'Electronic/Dance',
    'electro-disco': 'Electronic/Dance', 'hi-nrg': 'Electronic/Dance',
    'freestyle': 'Electronic/Dance', 'synthwave': 'Electronic/Dance',
    'vaporwave': 'Electronic/Dance', 'future bass': 'Electronic/Dance',
    'bass music': 'Electronic/Dance', 'trap edm': 'Electronic/Dance',
    'moombahton': 'Electronic/Dance', 'jersey club': 'Electronic/Dance',
    'ghettotech': 'Electronic/Dance', 'eurodance': 'Electronic/Dance',
    'eurobeat': 'Electronic/Dance', 'italo dance': 'Electronic/Dance',
    'bubblegum dance': 'Electronic/Dance', 'new beat': 'Electronic/Dance',
    'electroclash': 'Electronic/Dance', 'new rave': 'Electronic/Dance',
    'electro swing': 'Electronic/Dance', 'complextro': 'Electronic/Dance',
    'future rave': 'Electronic/Dance', 'industrial': 'Electronic/Dance',
    'ebm': 'Electronic/Dance', 'martial industrial': 'Electronic/Dance',
    'dark wave': 'Electronic/Dance', 'witch house': 'Electronic/Dance',
    'futurepop': 'Electronic/Dance', 'berlin school': 'Electronic/Dance',
    'minimal synth': 'Electronic/Dance', 'progressive electronic': 'Electronic/Dance',
    'bitpop': 'Electronic/Dance', 'ethereal wave': 'Electronic/Dance',
    'hip house': 'Electronic/Dance', 'alternative dance': 'Electronic/Dance',
    'folktronica': 'Electronic/Dance', 'funktronica': 'Electronic/Dance',
    'electropunk': 'Electronic/Dance', 'indietronica': 'Electronic/Dance',
    'leftfield': 'Electronic/Dance', 'breaks': 'Electronic/Dance',
    'hardstyle': 'Electronic/Dance', 'dungeon synth': 'Electronic/Dance',
    'lolicore': 'Electronic/Dance', 'chiptune': 'Electronic/Dance',
    'italo-disco': 'Electronic/Dance', 'electroacoustic': 'Experimental/Avant-Garde',
    # POP
    'pop': 'Pop', 'pop rock': 'Pop', 'pop soul': 'Pop', 'dance-pop': 'Pop',
    'electropop': 'Pop', 'teen pop': 'Pop', 'bubblegum pop': 'Pop',
    'power pop': 'Pop', 'baroque pop': 'Pop', 'art pop': 'Pop',
    'chamber pop': 'Pop', 'indie pop': 'Pop', 'noise pop': 'Pop',
    'sunshine pop': 'Pop', 'sophisti-pop': 'Pop', 'brill building': 'Pop',
    'space age pop': 'Pop', 'europop': 'Pop', 'j-pop': 'Pop', 'k-pop': 'Pop',
    'q-pop': 'Pop', 'alternative pop': 'Pop', 'ambient pop': 'Pop',
    'psychedelic pop': 'Pop', 'progressive pop': 'Pop', 'avant-garde pop': 'Pop',
    'traditional pop': 'Pop', 'schlager': 'Pop', 'yé-yé': 'Pop',
    'pop yeh-yeh': 'Pop', 'shibuya-kei': 'Pop', 'ballad': 'Pop',
    'synth-pop': 'Pop', 'new romantic': 'Pop', 'new wave': 'Pop',
    'hyperpop': 'Pop', 'glitch pop': 'Pop', 'hypnagogic pop': 'Pop',
    'bedroom pop': 'Pop', 'twee pop': 'Pop', 'folk pop': 'Pop',
    # HIP HOP/RAP
    'hip hop': 'Hip Hop/Rap', 'rap': 'Hip Hop/Rap',
    'east coast hip hop': 'Hip Hop/Rap', 'west coast hip hop': 'Hip Hop/Rap',
    'southern hip hop': 'Hip Hop/Rap', 'dirty south': 'Hip Hop/Rap',
    'gangsta rap': 'Hip Hop/Rap', 'g-funk': 'Hip Hop/Rap',
    'hardcore hip hop': 'Hip Hop/Rap', 'conscious hip hop': 'Hip Hop/Rap',
    'political hip hop': 'Hip Hop/Rap', 'alternative hip hop': 'Hip Hop/Rap',
    'experimental hip hop': 'Hip Hop/Rap', 'abstract hip hop': 'Hip Hop/Rap',
    'instrumental hip hop': 'Hip Hop/Rap', 'drumless hip hop': 'Hip Hop/Rap',
    'boom bap': 'Hip Hop/Rap', 'jazz rap': 'Hip Hop/Rap', 'trap': 'Hip Hop/Rap',
    'drill': 'Hip Hop/Rap', 'chicago drill': 'Hip Hop/Rap',
    'jersey drill': 'Hip Hop/Rap', 'detroit trap': 'Hip Hop/Rap',
    'grime': 'Hip Hop/Rap', 'crunk': 'Hip Hop/Rap', 'crunkcore': 'Hip Hop/Rap',
    'snap': 'Hip Hop/Rap', 'hyphy': 'Hip Hop/Rap', 'cloud rap': 'Hip Hop/Rap',
    'emo rap': 'Hip Hop/Rap', 'plugg': 'Hip Hop/Rap', 'rage': 'Hip Hop/Rap',
    'miami bass': 'Hip Hop/Rap', 'memphis rap': 'Hip Hop/Rap',
    'chopped and screwed': 'Hip Hop/Rap', 'horrorcore': 'Hip Hop/Rap',
    'nerdcore': 'Hip Hop/Rap', 'underground hip hop': 'Hip Hop/Rap',
    'industrial hip hop': 'Hip Hop/Rap', 'turntablism': 'Hip Hop/Rap',
    'lo-fi hip hop': 'Hip Hop/Rap', 'ratchet music': 'Hip Hop/Rap',
    'digicore': 'Hip Hop/Rap', 'pop rap': 'Hip Hop/Rap',
    'christian hip hop': 'Hip Hop/Rap', 'country rap': 'Hip Hop/Rap',
    # R&B/SOUL/FUNK
    'r&b': 'R&B/Soul/Funk', 'rhythm & blues': 'R&B/Soul/Funk',
    'contemporary r&b': 'R&B/Soul/Funk', 'alternative r&b': 'R&B/Soul/Funk',
    'neo soul': 'R&B/Soul/Funk', 'quiet storm': 'R&B/Soul/Funk',
    'soul': 'R&B/Soul/Funk', 'southern soul': 'R&B/Soul/Funk',
    'northern soul': 'R&B/Soul/Funk', 'deep soul': 'R&B/Soul/Funk',
    'psychedelic soul': 'R&B/Soul/Funk', 'progressive soul': 'R&B/Soul/Funk',
    'blue-eyed soul': 'R&B/Soul/Funk', 'smooth soul': 'R&B/Soul/Funk',
    'chicago soul': 'R&B/Soul/Funk', 'philly soul': 'R&B/Soul/Funk',
    'motown': 'R&B/Soul/Funk', 'doo-wop': 'R&B/Soul/Funk',
    'funk': 'R&B/Soul/Funk', 'p-funk': 'R&B/Soul/Funk',
    'synth funk': 'R&B/Soul/Funk', 'boogie': 'R&B/Soul/Funk',
    'go-go': 'R&B/Soul/Funk', 'new orleans r&b': 'R&B/Soul/Funk',
    'minneapolis sound': 'R&B/Soul/Funk', 'electro-funk': 'R&B/Soul/Funk',
    'hip hop soul': 'R&B/Soul/Funk', 'new jack swing': 'R&B/Soul/Funk',
    'trap soul': 'R&B/Soul/Funk',
    # COUNTRY/AMERICANA
    'country': 'Country/Americana', 'classic country': 'Country/Americana',
    'traditional country': 'Country/Americana', 'contemporary country': 'Country/Americana',
    'country pop': 'Country/Americana', 'country soul': 'Country/Americana',
    'outlaw country': 'Country/Americana', 'alternative country': 'Country/Americana',
    'progressive country': 'Country/Americana', 'neo-traditional country': 'Country/Americana',
    'bakersfield sound': 'Country/Americana', 'nashville sound': 'Country/Americana',
    'honky tonk': 'Country/Americana', 'western swing': 'Country/Americana',
    'bluegrass': 'Country/Americana', 'progressive bluegrass': 'Country/Americana',
    'bro-country': 'Country/Americana', 'red dirt': 'Country/Americana',
    'urban cowboy': 'Country/Americana', 'country yodeling': 'Country/Americana',
    'yodeling': 'Country/Americana', 'americana': 'Country/Americana',
    'cajun': 'Country/Americana', 'zydeco': 'Country/Americana',
    'swamp pop': 'Country/Americana', 'old-time': 'Country/Americana',
    'texas country': 'Country/Americana', 'western': 'Country/Americana',
    # FOLK
    'folk': 'Folk', 'contemporary folk': 'Folk', 'progressive folk': 'Folk',
    'psychedelic folk': 'Folk', 'chamber folk': 'Folk', 'avant-folk': 'Folk',
    'anti-folk': 'Folk', 'indie folk': 'Folk', 'neofolk': 'Folk',
    'neo-acoustic': 'Folk', 'stomp and holler': 'Folk', 'singer-songwriter': 'Folk',
    'ambient americana': 'Folk', 'irish folk': 'Folk', 'alternative folk': 'Folk',
    'freak folk': 'Folk', 'contra': 'Folk', 'country folk': 'Folk',
    # JAZZ
    'jazz': 'Jazz', 'bebop': 'Jazz', 'hard bop': 'Jazz', 'post-bop': 'Jazz',
    'cool jazz': 'Jazz', 'free jazz': 'Jazz', 'avant-garde jazz': 'Jazz',
    'jazz fusion': 'Jazz', 'jazz rock': 'Jazz', 'jazz pop': 'Jazz',
    'contemporary jazz': 'Jazz', 'smooth jazz': 'Jazz', 'swing': 'Jazz',
    'big band': 'Jazz', 'dixieland': 'Jazz', 'acid jazz': 'Jazz',
    'spiritual jazz': 'Jazz', 'vocal jazz': 'Jazz', 'digital fusion': 'Jazz',
    'jazz-funk': 'Jazz', 'soul jazz': 'Jazz', 'close harmony': 'Jazz',
    'latin jazz': 'Jazz', 'nu jazz': 'Jazz', 'modal jazz': 'Jazz',
    'free improvisation': 'Jazz',
    # BLUES
    'blues': 'Blues', 'electric blues': 'Blues', 'chicago blues': 'Blues',
    'delta blues': 'Blues', 'acoustic blues': 'Blues', 'texas blues': 'Blues',
    'electric texas blues': 'Blues', 'hill country blues': 'Blues',
    'classic blues': 'Blues', 'piano blues': 'Blues', 'british blues': 'Blues',
    'british rhythm & blues': 'Blues', 'soul blues': 'Blues',
    'boogie-woogie': 'Blues', 'country blues': 'Blues',
    # METAL
    'metal': 'Metal', 'heavy metal': 'Metal', 'thrash metal': 'Metal',
    'death metal': 'Metal', 'black metal': 'Metal', 'power metal': 'Metal',
    'speed metal': 'Metal', 'progressive metal': 'Metal', 'folk metal': 'Metal',
    'gothic metal': 'Metal', 'symphonic metal': 'Metal', 'glam metal': 'Metal',
    'nu metal': 'Metal', 'metalcore': 'Metal', 'deathcore': 'Metal',
    'grindcore': 'Metal', 'goregrind': 'Metal', 'melodic death metal': 'Metal',
    'melodic black metal': 'Metal', 'atmospheric black metal': 'Metal',
    'depressive black metal': 'Metal', 'blackgaze': 'Metal',
    'symphonic black metal': 'Metal', 'brutal death metal': 'Metal',
    'technical death metal': 'Metal', 'death-doom metal': 'Metal',
    'funeral doom metal': 'Metal', 'traditional doom metal': 'Metal',
    'doom metal': 'Metal', 'sludge metal': 'Metal', 'stoner metal': 'Metal',
    'groove metal': 'Metal', 'industrial metal': 'Metal',
    'neoclassical metal': 'Metal', 'nwobhm': 'Metal', 'mathcore': 'Metal',
    'djent': 'Metal', 'kawaii metal': 'Metal', 'alternative metal': 'Metal',
    'crossover thrash': 'Metal', 'noisecore': 'Metal', 'post-metal': 'Metal',
    'funk metal': 'Metal', 'rap metal': 'Metal', 'rapcore': 'Metal',
    'trap metal': 'Metal', 'mashcore': 'Metal', 'melodic metalcore': 'Metal',
    'avant-garde metal': 'Metal', 'southern metal': 'Metal',
    'christian metal': 'Metal', 'progressive metalcore': 'Metal',
    'deathgrind': 'Metal', 'electro-industrial': 'Metal',
    'us power metal': 'Metal', 'electronicore': 'Metal',
    # PUNK/HARDCORE
    'punk': 'Punk/Hardcore', 'punk rock': 'Punk/Hardcore',
    'hardcore punk': 'Punk/Hardcore', 'post-hardcore': 'Punk/Hardcore',
    'anarcho-punk': 'Punk/Hardcore', 'd-beat': 'Punk/Hardcore',
    'skate punk': 'Punk/Hardcore', 'ska punk': 'Punk/Hardcore',
    'horror punk': 'Punk/Hardcore', 'glam punk': 'Punk/Hardcore',
    'art punk': 'Punk/Hardcore', 'proto-punk': 'Punk/Hardcore',
    'dance-punk': 'Punk/Hardcore', 'emo': 'Punk/Hardcore',
    'emocore': 'Punk/Hardcore', 'screamo': 'Punk/Hardcore',
    'emoviolence': 'Punk/Hardcore', 'grebo': 'Punk/Hardcore',
    'post-punk': 'Punk/Hardcore', 'post-punk revival': 'Punk/Hardcore',
    'no wave': 'Punk/Hardcore', 'midwest emo': 'Punk/Hardcore',
    'punk blues': 'Punk/Hardcore', 'pop punk': 'Punk/Hardcore',
    'emo pop': 'Punk/Hardcore', 'folk punk': 'Punk/Hardcore',
    'garage punk': 'Punk/Hardcore', 'melodic hardcore': 'Punk/Hardcore',
    'celtic punk': 'Punk/Hardcore', 'cowpunk': 'Punk/Hardcore',
    'neue deutsche welle': 'Punk/Hardcore', 'easycore': 'Punk/Hardcore',
    'crust punk': 'Punk/Hardcore',
    # LATIN
    'latin': 'Latin', 'latin ballad': 'Latin', 'latin pop': 'Latin',
    'latin rock': 'Latin', 'salsa': 'Latin', 'bachata': 'Latin',
    'mambo': 'Latin', 'son cubano': 'Latin', 'tango': 'Latin',
    'mariachi': 'Latin', 'norteño': 'Latin', 'corrido tumbado': 'Latin',
    'regional mexicano': 'Latin', 'reggaeton': 'Latin',
    'electro latino': 'Latin', 'canción melódica': 'Latin',
    'tex-mex': 'Latin', 'trap latino': 'Latin', 'bossa nova': 'Latin',
    'choro': 'Latin', 'mpb': 'Latin', 'banda sinaloense': 'Latin',
    'ranchera': 'Latin', 'bolero': 'Latin', 'merengue': 'Latin',
    'cumbia': 'Latin', 'corrido': 'Latin', 'vallenato': 'Latin',
    'sierreño': 'Latin',
    # REGGAE/CARIBBEAN
    'reggae': 'Reggae/Caribbean', 'roots reggae': 'Reggae/Caribbean',
    'dub': 'Reggae/Caribbean', 'lovers rock': 'Reggae/Caribbean',
    'ragga': 'Reggae/Caribbean', 'reggae-pop': 'Reggae/Caribbean',
    'ska': 'Reggae/Caribbean', 'rocksteady': 'Reggae/Caribbean',
    'third wave ska': 'Reggae/Caribbean', '2 tone': 'Reggae/Caribbean',
    'calypso': 'Reggae/Caribbean', 'soca': 'Reggae/Caribbean',
    'dancehall': 'Reggae/Caribbean',
    # CLASSICAL
    'classical': 'Classical', 'contemporary classical': 'Classical',
    'modern classical': 'Classical', 'cinematic classical': 'Classical',
    'post-minimalism': 'Classical', 'opera': 'Classical',
    'orchestral': 'Classical', 'baroque': 'Classical',
    'classical crossover': 'Classical',
    # GOSPEL/CHRISTIAN/RELIGIOUS
    'gospel': 'Gospel/Christian/Religious', 'contemporary gospel': 'Gospel/Christian/Religious',
    'southern gospel': 'Gospel/Christian/Religious', 'christian rock': 'Gospel/Christian/Religious',
    'contemporary christian': 'Gospel/Christian/Religious',
    'christmas music': 'Gospel/Christian/Religious',
    'country gospel': 'Gospel/Christian/Religious',
    'gospel reggae': 'Gospel/Christian/Religious',
    'praise & worship': 'Gospel/Christian/Religious',
    # WORLD MUSIC
    'world fusion': 'World Music', 'afrobeat': 'World Music',
    'afrobeats': 'World Music', 'afro-funk': 'World Music',
    'soukous': 'World Music', 'chalga': 'World Music',
    'maloya élektrik': 'World Music', 'filmi': 'World Music',
    'persian pop': 'World Music', 'amapiano': 'World Music',
    'néo-trad': 'World Music', 'hindustani classical': 'World Music',
    'indian classical': 'World Music', 'chanson française': 'World Music',
    'celtic': 'World Music', 'klezmer': 'World Music', 'fado': 'World Music',
    'flamenco': 'World Music', 'celtic new age': 'World Music',
    # EXPERIMENTAL/AVANT-GARDE
    'experimental': 'Experimental/Avant-Garde', 'avant-garde': 'Experimental/Avant-Garde',
    'experimental rock': 'Experimental/Avant-Garde', 'noise': 'Experimental/Avant-Garde',
    'harsh noise': 'Experimental/Avant-Garde', 'black noise': 'Experimental/Avant-Garde',
    'sound art': 'Experimental/Avant-Garde', 'sound collage': 'Experimental/Avant-Garde',
    'plunderphonics': 'Experimental/Avant-Garde', 'epic collage': 'Experimental/Avant-Garde',
    'spamwave': 'Experimental/Avant-Garde', 'dariacore': 'Experimental/Avant-Garde',
    'experimental electronic': 'Experimental/Avant-Garde', 'art rock': 'Experimental/Avant-Garde',
    'math rock': 'Experimental/Avant-Garde', 'avant-prog': 'Experimental/Avant-Garde',
    # EASY LISTENING/VOCAL
    'easy listening': 'Easy Listening/Vocal', 'lounge': 'Easy Listening/Vocal',
    'cocktail nation': 'Easy Listening/Vocal', 'comedy': 'Easy Listening/Vocal',
    'spoken word': 'Easy Listening/Vocal', "children's music": 'Easy Listening/Vocal',
    'instrumental': 'Easy Listening/Vocal', 'musical': 'Easy Listening/Vocal',
    'music hall': 'Easy Listening/Vocal', 'new age': 'Easy Listening/Vocal',
    'operatic pop': 'Easy Listening/Vocal', 'standup comedy': 'Easy Listening/Vocal',
    'poetry': 'Easy Listening/Vocal',
}

def get_umbrella_genre(tags):
    if not isinstance(tags, (list, tuple)):
        return None
    for tag in tags:
        mapped = musicbrainz_to_major_genre.get(tag.lower())
        if mapped:
            return mapped
    return None

df_mb["mb_genre"] = df_mb["mb_tags"].apply(get_umbrella_genre)

# ── Join to album catalog ────────────────────────────────────────────────────
album_catalog["artist_norm_clean"] = album_catalog["artist_norm"].apply(normalize)
album_catalog["album_norm_clean"]  = album_catalog["album_norm"].apply(normalize)

MB_COLS = [
    "mb_artist_norm", "mb_album_norm", "mb_release_group_mbid", "mb_artist_mbid",
    "release_year", "release_type", "artist_type", "artist_area",
    "label", "catalog_number", "mb_tags", "mb_genre"
]

df_enriched = album_catalog.merge(
    df_mb[MB_COLS],
    left_on=["artist_norm_clean", "album_norm_clean"],
    right_on=["mb_artist_norm", "mb_album_norm"],
    how="left"
)
df_enriched.drop(
    columns=["artist_norm_clean", "album_norm_clean", "mb_artist_norm", "mb_album_norm"],
    inplace=True
)

matched = df_enriched["mb_release_group_mbid"].notna().sum()
print(f"Match rate:          {matched:,}/{len(df_enriched):,} ({100*matched/len(df_enriched):.1f}%)")
print(f"With release year:   {df_enriched['release_year'].notna().sum():,}")
print(f"With label:          {df_enriched['label'].notna().sum():,}")
print(f"With umbrella genre: {df_enriched['mb_genre'].notna().sum():,}")

df_enriched.to_parquet(ENRICHED_PATH, index=False)
print(f"\nSaved: {ENRICHED_PATH}")
print("Upload album_catalog_enriched.parquet to Google Drive when done.")


Albums to enrich: 44,716
Release groups found: 622,268
Release groups with tags: 331,394
Match rate:          28,590/44,716 (63.9%)
With release year:   28,467
With label:          27,708
With umbrella genre: 23,849

Saved: /Users/jamesemcnally/Documents/GitHub/critical-listener/datasets/album_catalog_enriched.parquet
Upload album_catalog_enriched.parquet to Google Drive when done.
